# O.E.6 - Despliegue ML-Ops (Sprint 3)

**Objetivo especifico:** exportar el modelo a ONNX y demostrar el servicio REST funcional.

**Evidencia generada:** `models/model.onnx`, `reports/predictions.jsonl`.

**En una terminal:** `docker compose -f deployment/docker-compose.yml up --build` o `uvicorn deployment.app:app`.

In [ ]:
import sys, os
from pathlib import Path
root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
sys.path.insert(0, str(root))
os.chdir(root)
print('Raiz del proyecto:', root)

In [ ]:
# 1) Exportar a ONNX (portable para produccion)
from src.config import get_paths, load_config
from src.predict import export_onnx
paths = get_paths(load_config())
out = export_onnx(paths['best_model'], paths['onnx_model'])
print('ONNX:', out)

In [ ]:
# 2) Demo del endpoint /predict usando TestClient (sin levantar servidor)
import io
import numpy as np
from PIL import Image
from fastapi.testclient import TestClient
from deployment.app import create_app

client = TestClient(create_app(paths['best_model']))
img = (np.random.rand(32, 32, 3) * 255).astype('uint8')
buf = io.BytesIO(); Image.fromarray(img).save(buf, format='PNG'); buf.seek(0)
resp = client.post('/predict', files={'file': ('sample.png', buf, 'image/png')})
print('status:', resp.status_code)
print(resp.json())